# Models

In DBT, a model is a set of SQL code and configuration files that are executed in the target database. According to the configuration, the database would then create the essences.

## Properties

Alongside with the `.sql` file the model in dbt can be described with `.yml` file - **properties file**. This file contains metadata and configuration, and probably defines the tests for the data model produce.

---

The following cell defines the dbt project:

In [1]:
#init
dbt init properties --profile knowledge -q
cd properties

Next cells define the simple model a sample properties file to it:

In [2]:
# file properties/models/some_query.sql
select * from (
    values
    (10, 'line1'),
    (20, 'line2')
) as t(id, line)

In [5]:
# file properties/models/some_query.yml
version: 2

models:
    - name: some_query
      description: The example query
      columns:
        - name: id
          description: Some description of the id
        - name: line
          description: Some description of the line

### Contract

If the `config.contract.enforced` field of the model is set to the `true`:

```yml
config:
    contract:
        enforced: true
```

The dbt will check if the data corresponds to the configuration defined in the properties before the materialization of the table.

---

The following cell defines the example project:

In [7]:
#init
dbt init contract --profile knowledge -q
cd contract

Some kind of sql code implementing the model:

In [18]:
# file contract/models/some_query.sql
select * from (
    values
    (10, 'line1'),
    (20, 'line2')
) as t(id, line)

The properties of the model:

In [24]:
# file contract/models/some_query.yml
version: 2

models:
    - name: some_query
      description: The example query
      config:
          contract:
              enforced: true
      columns:
        - name: id
          description: Some description of the id
          data_type: int
        - name: line
          description: Some description of the line
          data_type: int

Note that to the `line` column has been assigned the `int` data type, even though it's clear from the query that it takes a `string` data type.

The attempt to materialize this model ends with the corresponding error from dbt:

In [25]:
dbt run -q

08:24:55  2 of 3 ERROR creating sql view model public.some_query ......................... [ERROR in 0.19s]
08:24:55  Failure in model some_query (models/some_query.sql)
08:24:55    Compilation Error in model some_query (models/some_query.sql)
  This model has an enforced contract that failed.
  Please ensure the name, data_type, and number of columns in your contract match the columns in your model's definition.
  
  | column_name | definition_type | contract_type | mismatch_reason    |
  | ----------- | --------------- | ------------- | ------------------ |
  | line        | TEXT            | INTEGER       | data type mismatch |
  
  
  > in macro assert_columns_equivalent (macros/relations/column/columns_spec_ddl.sql)
  > called by macro default__get_assert_columns_equivalent (macros/relations/column/columns_spec_ddl.sql)
  > called by macro get_assert_columns_equivalent (macros/relations/column/columns_spec_ddl.sql)
  > called by macro default__create_view_as (macros/relations/view

: 1

## Materialization

The same model can be represented in a dabase using different database mechanisms. At first it can be presented as: table, view or materialized view. Also in dbt you can manage the way the essesnce is builded to the database (build in this context is the process of putting data to the warehouse). Different approaches are implemented through different materialization approaches:

- Table.
- View.
- Incremental.
- Ephemeral.
- Materialized view.

## Incremental

Incremental materialisation is relativly complex and has many configuration options. Therefore, it is considered as a separate section.

The incremental model was created as a table in the database. However, different runs of the same model only update the records that require an update.

The incremental materialization can be implemeted using different database mechanisms, so there are different incremental strategies:

- Append.
- Delete + Insert.

**Note** you can implement a custom strategy as well.

Check [About incremental models](https://docs.getdbt.com/docs/build/incremental-models-overview?version=1.12) page of the dbt documentation.

---

Consider a simple example of the materialization. The incremental model loads data from the seed but only takes data only with new categories.

The following cells initializes the project with the corresponding seeds:

In [1]:
# init
dbt init incremental_example --profile knowledge -q
cd incremental_example

In [2]:
# file incremental_example/seeds/data.csv
category,value
'a',20
'a',40

In [3]:
dbt seed -q

The implementation of the model:

In [4]:
# file incremental_example/models/my_incremental_model.sql
{{
    config(
        materialized='incremental'
    )
}}

select *
from {{ ref('data') }} as source_data

{% if is_incremental() %}

where not exists (
    select 1
    from {{ this }} as target_data
    where target_data.category = source_data.category
)

{% endif %}

The following cell runs the model:

In [5]:
dbt run -q --full-refresh --select my_incremental_model
dbt show -q --inline "select * from public.my_incremental_model"

| category | value |
| -------- | ----- |
| 'a'      |    20 |
| 'a'      |    40 |



The following cell updates the model.

**Note** that a new record was added to the data with the category set to "a".

In [6]:
# file incremental_example/seeds/data.csv
category,value
'a',20
'a',40
'a',-200
'b',50
'b',70

In [7]:
dbt seed -q
dbt run -q --select my_incremental_model
dbt show -q --inline "select * from public.my_incremental_model"

| category | value |
| -------- | ----- |
| 'a'      |    20 |
| 'a'      |    40 |
| 'b'      |    50 |
| 'b'      |    70 |



The model materialization only included records with a category that wasn't represented in the previous insert, which is according to the logic of the incremental model.